In [2]:
import pandas as pd
import numpy as np

# ============================================================
# 1. LOAD RAW DATASET
# ============================================================

df = pd.read_csv("datasets/drilling_data_ml_ready.csv") 

print("Original shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Original shape: (600359, 36)

Columns:
['block_position', 'weight_on_bit', 'hookload', 'slips_set', 'rop_depth_hour', 'on_bottom', 'top_drive_rpm', 'top_drive_torque_ft_lbs', 'flow_in', 'pump_pressure', 'spm_total', 'pit_volume_active', 'pit_gl_active', 'gas_total_units', 'trip_volume_active', 'trip_gl', 'return_flow', 'rig_mode', 'rockit_on_off', 'mwd_inclination', 'mwd_azimuth', 'mud_temp', 'h2s_01', 'rig_event_code', 'total_depth', 'bit_diameter', 'bit_rpm', 'depth_hole_tvd', 'differential_pressure', 'downhole_torque', 'drill_mode', 'year', 'month', 'day', 'hour', 'day_of_week']


In [3]:
# ============================================================
# 2. BASIC DATA QUALITY CHECK
# ============================================================

print("Shape:", df.shape)

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:")
print(df.duplicated().sum())

Shape: (600359, 36)

Data types:
block_position             float64
weight_on_bit              float64
hookload                   float64
slips_set                  float64
rop_depth_hour             float64
on_bottom                  float64
top_drive_rpm              float64
top_drive_torque_ft_lbs    float64
flow_in                    float64
pump_pressure              float64
spm_total                  float64
pit_volume_active          float64
pit_gl_active              float64
gas_total_units            float64
trip_volume_active         float64
trip_gl                    float64
return_flow                float64
rig_mode                   float64
rockit_on_off              float64
mwd_inclination            float64
mwd_azimuth                float64
mud_temp                   float64
h2s_01                     float64
rig_event_code             float64
total_depth                float64
bit_diameter               float64
bit_rpm                    float64
depth_hole_tvd        

In [4]:
# ============================================================
# 3. IDENTIFY SENTINEL VALUES
# ============================================================

sentinel_value = -999.25

print("Occurrences of -999.25:")

for col in df.select_dtypes(include=np.number).columns:
    count = (df[col] == sentinel_value).sum()

    if count > 0:
        print(f"{col}: {count}")

Occurrences of -999.25:
block_position: 216
slips_set: 236
on_bottom: 215
top_drive_torque_ft_lbs: 216
gas_total_units: 46277
rig_mode: 46
rockit_on_off: 201163
mwd_inclination: 51642
mwd_azimuth: 51647
mud_temp: 1694
h2s_01: 23231
rig_event_code: 46
differential_pressure: 216
drill_mode: 216


In [5]:
# ============================================================
# 4. CONVERT SENTINEL VALUES TO NaN
# ============================================================

df.replace(-999.25, np.nan, inplace=True)

print("Remaining -999.25 values:",
      (df.select_dtypes(include=np.number) == -999.25).sum().sum())

Remaining -999.25 values: 0


In [6]:
# ============================================================
# 5. REMOVE DUPLICATE ROWS
# ============================================================

duplicates_before = df.duplicated().sum()

print("Duplicate rows found:", duplicates_before)

df.drop_duplicates(inplace=True)

df.reset_index(drop=True, inplace=True)

print("Shape after duplicate removal:", df.shape)

Duplicate rows found: 1650
Shape after duplicate removal: (598709, 36)


In [7]:
# ============================================================
# 6. CHECK INFINITE VALUES
# ============================================================

numeric_cols = df.select_dtypes(include=np.number).columns

inf_counts = np.isinf(df[numeric_cols]).sum()

print("Columns containing infinite values:")

print(inf_counts[inf_counts > 0])

Columns containing infinite values:
Series([], dtype: int64)


In [8]:
# ============================================================
# 7. TIME COLUMN VALIDATION
# ============================================================

time_cols = [
    'year',
    'month',
    'day',
    'hour',
    'day_of_week'
]

for col in time_cols:
    print(
        col,
        "Min:", df[col].min(),
        "Max:", df[col].max(),
        "Missing:", df[col].isnull().sum()
    )

year Min: 2020 Max: 2021 Missing: 0
month Min: 1 Max: 12 Missing: 0
day Min: 1 Max: 31 Missing: 0
hour Min: 0 Max: 23 Missing: 0
day_of_week Min: 0 Max: 6 Missing: 0


In [9]:
df['timestamp'] = pd.to_datetime(
    dict(
        year=df['year'],
        month=df['month'],
        day=df['day'],
        hour=df['hour']
    ),
    errors='coerce'
)

print("Invalid timestamps:", df['timestamp'].isnull().sum())

Invalid timestamps: 0


In [10]:
# ============================================================
# 8. FINAL DATA QUALITY REPORT
# ============================================================

print("=" * 60)
print("FINAL DATA QUALITY REPORT")
print("=" * 60)

print("Final shape:", df.shape)

print("\nTotal missing values:")
print(df.isnull().sum().sum())

print("\nColumns containing missing values:")

missing = df.isnull().sum()

print(
    missing[missing > 0]
    .sort_values(ascending=False)
)

print("\nDuplicate rows:")
print(df.duplicated().sum())

print("\nInfinite values:")
print(
    np.isinf(
        df.select_dtypes(include=np.number)
    ).sum().sum()
)

FINAL DATA QUALITY REPORT
Final shape: (598709, 37)

Total missing values:
372635

Columns containing missing values:
rockit_on_off              199513
mwd_azimuth                 51618
mwd_inclination             51613
gas_total_units             46269
h2s_01                      23231
slips_set                      65
mud_temp                       46
block_position                 45
top_drive_torque_ft_lbs        45
differential_pressure          45
drill_mode                     45
on_bottom                      44
rig_mode                       28
rig_event_code                 28
dtype: int64

Duplicate rows:
0

Infinite values:
0


In [11]:
# ============================================================
# 9. SAVE MASTER CLEANED DATASET
# ============================================================

df.to_csv(
    "drilling_data_cleaned.csv",
    index=False
)

print("Dataset saved successfully!")
print("Final shape:", df.shape)

Dataset saved successfully!
Final shape: (598709, 37)
